**TAISS 2026 - Filière F1 (Data Science)**

**TP2 -  Segmentation RFM**

**Partie 1 :** Nettoyage des transactions

**Objectif de ce notebook :** partir du fichier brut et produire un jeu de transactions propre et traçable, qui servira de base au calcul des features RFM en Partie 2. 

**Question de rapport n°1:** Les annulations et retours changent-ils significativement les segments ? Comment le vérifier ?
Ce notebook prépare deux jeux de données (avec et sans compensation des retours) précisément pour pouvoir répondre par comparaison.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Configuration de l'affichage Pandas et du style des graphiques
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
sns.set_theme(style="whitegrid")

# Définition des dossiers du projet
RAW = Path("../data/raw/online_retail_II.xlsx")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)

**1.1 Chargement et concaténation des deux feuilles**

Le fichier Excel contient deux exercices annuels. On les concatène en ajoutant une colonne `Source` pour pouvoir vérifier plus tard qu'aucun des deux millésimes n'est surreprésenté après nettoyage.

In [ ]:
%%time
# Import du fichier Excel contenant plusieurs feuilles
raw_xl = pd.ExcelFile(RAW)
print("Feuilles :", xl.sheet_names)

# Lire et fusionner toutes les feuilles dans un seul DataFrame
frames = []
for sheet in xl.sheet_names:
    d = xl.parse(sheet)
    d["Source"] = sheet
    print(f"  {sheet}: {d.shape}")
    frames.append(d)

df = pd.concat(frames, ignore_index=True)

# Convertir 'Customer ID' en 'CustomerID'
df = df.rename(columns={"Customer ID": "CustomerID"})

n0 = len(df)
print("Total brut :", f"{n:,} lignes")
df.head()

In [ ]:
%%time
# Afficher les informations générales sur les colonnes et les types de données
df.info()

print("\nValeurs manquantes par colonne :")
print(df.isna().sum())

print("\nPériode couverte par les données :", df.InvoiceDate.min(), "-", df.InvoiceDate.max())
print("Nombre de clients uniques (brut) :", df.CustomerID.nunique())
print("Nombre de pays uniques :", df.Country.nunique())

**1.2 Journal de nettoyage** <br>
Créer une fonction qui enregistre chaque étape du nettoyage (nom, lignes restantes, lignes retirées) pour la traçabilité. 

In [ ]:
# Initialisation du journal pour enregistrer les étapes de nettoyage
journal = []
def etape(nom, d, commentaire=""):
    """Enregistre le nombre de lignes restantes après chaque étape de filtrage

    et affiche une ligne de synthèse.
    """
    # Récupère le nombre de lignes précédant cette étape (ou n0 au départ)
    prev = journal[-1]["restant"] if journal else n0

    # Calcule le nombre de lignes retirées et ajoute l'étape au journal
    journal.append(
        {
            "etape": nom,
            "restant": len(d),
            "retirees": prev - len(d),
            "commentaire": commentaire,
        }
    )

    # Affiche un résumé aligné : [Nom de l'étape] -[Lignes retirées] -> [Lignes restantes]
    print(f"{nom:38s} -{prev-len(d):>8,} Lignes retirées  →  {len(d):>9,} lignes restantes")

    return d

---
**a) Doublons exacts :** 
Suppression des lignes strictement identiques (même facture, même produit, même quantité, même horodatage)

In [ ]:
COLS_METIER = ["Invoice", "StockCode", "Description", "Quantity",
               "InvoiceDate", "Price", "CustomerID", "Country"]
f1, f2 = xl.sheet_names
print("Chevauchement des feuilles :")
for s in xl.sheet_names:
    d = df[df.Source == s]
    print(f"  {s}: {d.InvoiceDate.min().date()} → {d.InvoiceDate.max().date()}")

communes = set(df.loc[df.Source == f1, "Invoice"]) & set(df.loc[df.Source == f2, "Invoice"])
print("  factures présentes dans les deux feuilles :", len(communes))

dup_total  = df.duplicated(subset=COLS_METIER).sum()
dup_intra  = (df[df.Source == f1].duplicated(subset=COLS_METIER).sum()
              + df[df.Source == f2].duplicated(subset=COLS_METIER).sum())
print(f"  doublons intra-feuille : {dup_intra:,} | dus au chevauchement : {dup_total - dup_intra:,}")

In [ ]:
# Les deux feuilles se CHEVAUCHENT sur le 01→09 déc. 2010.
# On dédoublonne sur les colonnes MÉTIER, en excluant 'Source' : sinon les deux copies

df = etape("doublons exacts", df.drop_duplicates(subset=COLS_METIER),
           "artefacts d'export + chevauchement des 2 feuilles")

---
**b) Lignes qui ne sont pas des ventes de produits :** 

Certains `StockCode` ne désignent pas un article mais des frais ou des écritures comptables :`POST` (port), `DOT` (DOTCOM postage), `M` (manual), `BANK CHARGES`, `AMAZONFEE`, `ADJUST`,`D` (discount), `S` (samples), `CRUK` (don), `TEST001/2`, `GIFT_...` (bons cadeaux).

On les retire du calcul RFM car le montant *Monetary* doit refléter la valeur des produits achetés, pas des frais de port ni des régularisations comptables. Un port de 18 £ ne dit rien de l'appétence produit d'un client.

In [ ]:
# 1. Normalisation des chaînes de caractères (majuscules et suppression des espaces superflus)
df["StockCode"] = df["StockCode"].astype(str).str.upper().str.strip()
df["Invoice"] = df["Invoice"].astype(str).str.upper().str.strip()

# 2. Liste des codes d'articles non-produits (frais, ajustements, tests, etc.)
NON_PRODUITS = {
    "POST",
    "DOT",
    "M",
    "C2",
    "D",
    "S",
    "BANK CHARGES",
    "ADJUST",
    "AMAZONFEE",
    "CRUK",
    "PADS",
    "B",
    "TEST001",
    "TEST002",
}

# 3. Masque pour identifier les lignes non-produit
masque_np = df["StockCode"].isin(NON_PRODUITS) | df["StockCode"].str.startswith(
    "GIFT_"
)

# 4. Affichage de la répartition des lignes non-produit
print("Répartition des lignes non-produit :")
print(df.loc[masque_np, "StockCode"].value_counts().head(15))

# 5. Application du filtre et mise à jour du journal
df = etape("lignes non-produit (frais, ajust.)", df[~masque_np])

---
**c) Séparation des annulations** <br>
Les factures commençant par `C` sont des **annulations / retours** : quantité négative, montant négatif. Elles ne sont pas du bruit, ce sont de vraies informations métier. On les **met de côté** dans `retours` au lieu de les jeter, pour pouvoir construire plus tard la variante « RFM net des retours » et répondre à la question 1 du rapport.

In [ ]:
# 1. Identifier les factures d'annulation commençant par "C" (Cancelled)
df["is_cancel"] = df["Invoice"].str.startswith("C")

# 2. Isoler les transactions de retour dans un DataFrame dédié
retours = df[df["is_cancel"]].copy()

# 3. Affichage des statistiques sur les annulations
print("Lignes d'annulation :", f"{len(retours):,}")
print(
    "Montant total retourné : £",
    f"{(retours['Quantity'] * retours['Price']).sum():,.0f}",
)
print("Clients concernés :", retours["CustomerID"].nunique())

# 4. Exclure les annulations du jeu de données principal et enregistrer l'étape dans le journal
df = etape("annulations mises de côté", df[~df["is_cancel"]])

---
**d) CustomerID manquant** 

In [ ]:
# 1. Calcul du chiffre d'affaires (CA) associé aux commandes sans identifiant client (CustomerID manquant)
part_ca_perdue = (
    df.loc[df.CustomerID.isna(), "Quantity"]
    * df.loc[df.CustomerID.isna(), "Price"]
).sum()

# 2. Calcul du chiffre d'affaires total du dataset courant
part_ca_total = (df["Quantity"] * df["Price"]).sum()

# 3. Affichage de la part de CA qui ne peut pas être attribuée à un client spécifique
print(
    f"CA sans CustomerID : £{part_ca_perdue:,.0f} soit {100 * part_ca_perdue / part_ca_total:.1f} % du CA restant"
)

# 4. Suppression des transactions anonymes et enregistrement de l'étape dans le journal
df = etape(
    "CustomerID manquant", df[df.CustomerID.notna()], "non rattachable à un client"
)

# 5. Conversion du CustomerID en entier (int) pour éliminer les décimales 
df["CustomerID"] = df["CustomerID"].astype(int)

---
**e) Quantités et prix non valides** 
Après retrait des annulations, il subsiste quelques lignes à `Quantity <= 0` ou `Price <= 0`(saisies erronées, articles offerts à 0 £). 
On les retire : un prix nul fausse le *Monetary* sans correspondre à une transaction commerciale.

In [ ]:
# 1. Vérifier le nombre de lignes présentant des quantités ou des prix nuls ou négatifs
print(
    "Quantity <= 0 :",
    (df["Quantity"] <= 0).sum(),
    " | Price <= 0 :",
    (df["Price"] <= 0).sum(),
)

# 2. Conserver uniquement les transactions avec une quantité strictement positive et un prix strictement positif
df = etape(
    "Quantity<=0 ou Price<=0", df[(df["Quantity"] > 0) & (df["Price"] > 0)]
)

---
**f) Montant par ligne et valeurs extrêmes:** 
Valeur extreme ici ne signifie pas forcement outlier

In [ ]:
# 1. Calcul de la colonne "Amount" (Montant total par ligne d'article)
df["Amount"] = df["Quantity"] * df["Price"]

# 2. Affichage des statistiques descriptives de la colonne Amount
print("Distribution du montant des lignes (Amount) :")
print(df["Amount"].describe(percentiles=[0.5, 0.9, 0.99, 0.999]))

print("\nTop 10 des lignes d'achats les plus élevées :")

# 3. Extraire et afficher les 10 plus grandes transactions selon la colonne Amount
display(
    df.nlargest(10, "Amount")[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "Amount",
            "CustomerID",
            "Country",
        ]
    ]
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# 1. Création d'une figure à 2 sous-graphiques côte à côte
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 2. Graphique de gauche : Distribution brute de la variable Amount
sns.histplot(df["Amount"], bins=100, ax=axes[0])
axes[0].set_title("Amount — échelle brute")

# 3. Graphique de droite : Distribution après transformation logarithmique log1p
# log1p(x) calcule log(1 + x) pour éviter l'erreur si x = 0
sns.histplot(np.log1p(df["Amount"]), bins=100, ax=axes[1])
axes[1].set_title("log1p(Amount)")

# 4. Ajustement des marges et sauvegarde de l'image
plt.tight_layout()
plt.savefig(FIG / "01_distribution_amount.png", dpi=150)
plt.show()

---
**g) Contrôles de cohérence finaux**

In [ ]:
# 1. Assertions de sécurité (S'assurer que le nettoyage est irréprochable avant la suite)
assert (
    df["CustomerID"].notna().all()
), "Erreur : il reste des identifiants clients manquants !"
assert (df["Quantity"] > 0).all() and (
    df["Price"] > 0
).all(), "Erreur : des valeurs de quantité ou de prix sont <= 0 !"
assert df["InvoiceDate"].between(
    "2009-12-01", "2011-12-31"
).all(), "Erreur : la période dépasse les dates limites autorisées !"

# 2. Affichage des métriques clés du jeu de données nettoyé
print("Lignes            :", f"{len(df):,}")
print("Clients uniques   :", df["CustomerID"].nunique())
print("Factures uniques  :", df["Invoice"].nunique())
print("Pays              :", df["Country"].nunique())
print("CA total          : £", f"{df['Amount'].sum():,.0f}")
print(
    "Période           :",
    df["InvoiceDate"].min().date(),
    "→",
    df["InvoiceDate"].max().date(),
)

# 3. Calcul et affichage des 8 principaux pays en termes de Chiffre d'Affaires (en %)
print("\nRépartition par pays (top 8) :")
print(
    (100 * df.groupby("Country")["Amount"].sum() / df["Amount"].sum())
    .nlargest(8)
    .round(1)
)

---
**1.3 Récapitulatif du nettoyage** 

In [ ]:
# 1. Conversion de la liste de suivi (journal) en DataFrame Pandas
recap = pd.DataFrame(journal)

# 2. Calcul du pourcentage de lignes retirées par rapport au volume brut initial (n0)
recap["% du brut retiré"] = (100 * recap["retirees"] / n0).round(2)

# 3. Affichage du tableau récapitulatif complet dans le notebook
display(recap)

# 4. Affichage du bilan global du nettoyage (nombre de lignes conservées)
print(
    f"\nBrut : {n0:,} lignes  →  Net : {len(df):,} lignes "
    f"({100 * len(df) / n0:.1f} % conservées)"
)

# 5. Sauvegarde du tableau de journalisation dans le dossier des données traitées (PROC)
recap.to_csv(PROC / "journal_nettoyage.csv", index=False)

---
**1.4 Variante « nette des retours »** <br>
Pour la question 1 du rapport, On construit un second jeu où les retours sont **compensés**.
les lignes d'annulation (quantités négatives) sont réintégrées et déduites du chiffre d'affaires client. En Partie 2 on calculera les RFM sur les **deux** jeux, et en Partie 3 on comparera les segmentations obtenues (ARI entre les deux partitions).

In [ ]:

# 1. Sélectionner uniquement les annulations qui possèdent un identifiant client (CustomerID présent)
retours_ok = retours[retours.CustomerID.notna()].copy()

# 2. Harmoniser le type du CustomerID en entier pour correspondre à df
retours_ok["CustomerID"] = retours_ok.CustomerID.astype(int)

# 3. Calculer le montant de la ligne de retour (Quantity étant négatif, Amount sera négatif)
retours_ok["Amount"] = retours_ok.Quantity * retours_ok.Price

# 4. Combiner le jeu d'achats propres (df) avec le jeu des retours clients (retours_ok)
df_net = pd.concat([df, retours_ok], ignore_index=True)

# 5. Affichage des statistiques comparatives entre les jeux de données
print("Jeu 'brut' (retours exclus) :", f"{len(df):,} lignes")
print("Jeu 'net'  (retours inclus) :", f"{len(df_net):,} lignes")

# 6. Calcul de la différence de Chiffre d'Affaires due à l'impact des annulations
print("Écart de CA : £", f"{df.Amount.sum() - df_net.Amount.sum():,.0f}")

---
**1.5 Sauvegarde des deux dataset** <br>
La Partie 2 repartira de ces fichiers.

In [ ]:
# 1. Ajout de la colonne 'Source' pour identifier l'origine de chaque enregistrement
df["Source"] = "achats"
df_net["Source"] = df_net["is_cancel"].map(
    {True: "annulation", False: "achats"}
)

# 2. Liste ordonnée des colonnes finales à conserver
cols = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "CustomerID",
    "Country",
    "Amount",
    "Source",
]

# 3. Export des deux versions du jeu de données nettoyé au format Parquet
df[cols].to_parquet(PROC / "transactions_clean.parquet", index=False)
df_net[cols].to_parquet(PROC / "transactions_clean_net.parquet", index=False)

# 4. Confirmation de la sauvegarde
print("Sauvegardé dans", PROC.resolve())

---
### **Partie 1 - Nettoyage Bref resumer** 
résumé Le jeu brut compte 1 067 371 lignes sur 2 feuilles (déc. 2009 - déc. 2011). Le nettoyage retire successivement les doublons exacts (34 335 lignes), les écritures non-produit (port, frais bancaires, ajustements - 5 815 lignes), les 19 104 lignes d'annulation (mises de côté pour une analyse comparative), les 243 007 lignes sans `CustomerID` (~23 % du volume, non rattachables à un client) et les quelques lignes à quantité ou prix nuls. Il reste **776 582 transactions, 5 852 clients, 36 597 factures et 17,1 M£ de CA** sur 41 pays. Les montants extrêmes sont conservés volontairement : ils correspondent aux commandes de grossistes que la segmentation doit isoler, et l'asymétrie sera traitée par transformation logarithmique en Partie 2.
